# Text Generation for Business Applications — T5 and GPT-2

**Course:** Natural Language Processing · Unit 5 — Language Models and Text Generation  
**Institution:** Universidad Tecnológica de Bolívar  
**Reference:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3rd ed.), Ch. 10 — Machine Translation and Encoder–Decoder Models  
**Python compatibility:** 3.10 · 3.11 · 3.12 · 3.13

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Load and compare **T5** (encoder–decoder) and **GPT-2** (decoder-only) from HuggingFace Hub
2. Apply T5 to **automatic report summarisation** with variable summary length
3. Use GPT-2 to **generate product descriptions** and control creativity with `temperature`
4. Automate **customer service responses** using conditional text generation
5. Compare decoding strategies: **greedy**, **beam search**, **top-k**, and **top-p sampling**
6. Combine generation with BLEU/ROUGE evaluation in an end-to-end pipeline

## Architecture Comparison

| Model | Architecture | Task style | Input format |
|---|---|---|---|
| **T5-small** | Encoder–Decoder | Seq2Seq (summarise, translate, Q&A) | `task_prefix: text` |
| **GPT-2** | Decoder-only | Auto-regressive continuation | Raw prompt text |


---

## 1. Environment Setup

Install dependencies **once** in your terminal:

```bash
pip install transformers torch evaluate sacrebleu rouge-score
```


In [ ]:
# pip install transformers torch evaluate sacrebleu rouge-score  # run once in terminal
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
)
import evaluate


---

## 2. Load T5 and GPT-2 Models

Models are downloaded from HuggingFace Hub on first run and cached locally. T5-small (~60M parameters) is fast for experimentation; use `t5-base` or `t5-large` for better quality.


In [ ]:
T5_MODEL = 't5-small'
GPT2_MODEL = 'gpt2'

t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL)

gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL)
gpt2_model = AutoModelForCausalLM.from_pretrained(GPT2_MODEL)

print(f'T5 loaded: {T5_MODEL}')
print(f'GPT-2 loaded: {GPT2_MODEL}')


> **Output interpretation:** Both models are cached in `~/.cache/huggingface/hub`. T5-small downloads ~231 MB; GPT-2 base ~548 MB. Subsequent runs use the local cache and are instant. If you are running on CPU, inference will be slower — add `.to('cuda')` if a GPU is available.


---

## 3. Business Case 1 — Automatic Report Summarisation with T5

**Scenario:** The executive team needs concise summaries of quarterly financial reports. T5 is fine-tuned for summarisation with the task prefix `'summarize: '`.


In [ ]:
financial_report = (
    'In the third quarter of 2024, Bancolombia reported net income of 2.3 trillion pesos, '
    'representing a 15% increase over the same period in 2023. '
    'The loan portfolio grew by 8%, driven primarily by commercial and mortgage segments. '
    'Non-performing loans remained stable at 3.2%, below the industry average. '
    'Digital channels processed 78% of all transactions, up from 65% the previous year. '
    'The board approved a dividend of 250 pesos per share, payable in December 2024.'
)

# T5 uses task prefixes to select the generation mode
t5_input = 'summarize: ' + financial_report
inputs = t5_tokenizer(t5_input, return_tensors='pt', max_length=512, truncation=True)

output_ids = t5_model.generate(
    **inputs,
    max_length=80,
    min_length=20,
    num_beams=4,
    early_stopping=True,
)
summary = t5_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(f'Original ({len(financial_report.split())} words):')
print(financial_report)
print(f'\nSummary ({len(summary.split())} words):')
print(summary)


> **Output interpretation:** Beam search (`num_beams=4`) explores 4 candidate sequences simultaneously and keeps the most likely one — typically producing more fluent, coherent summaries than greedy decoding. `min_length=20` prevents the model from producing a trivially short output. The compression ratio (original vs summary word count) indicates how aggressively the model condenses the content.


### 3.1 Effect of Summary Length

Varying `max_length` produces summaries at different levels of detail — useful for adapting the output to different audiences (executive one-liner vs. analyst paragraph).


In [ ]:
for max_len in (30, 60, 100):
    ids = t5_model.generate(
        **inputs, max_length=max_len, num_beams=4, early_stopping=True
    )
    text = t5_tokenizer.decode(ids[0], skip_special_tokens=True)
    print(f'max_length={max_len}: {text}')
    print()


> **Output interpretation:** Shorter `max_length` forces the model to be more selective — it retains only the highest-salience content. Longer `max_length` allows more detail but may introduce repetition if the source is already short. Choose a length proportional to the source document length (rule of thumb: target ~20% of source word count for a first-pass summary).


---

## 4. Business Case 2 — Product Description Generation with GPT-2

**Scenario:** An e-commerce team needs to auto-generate varied product descriptions from a keyword prompt. GPT-2's sampling mode (`do_sample=True`) produces creative, non-deterministic outputs.


In [ ]:
product_prompt = (
    'Product: Noise-cancelling wireless headphones\n'
    'Description:'
)

# Set pad_token_id to suppress a warning for GPT-2 (which has no official PAD token)
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
inputs_gpt2 = gpt2_tokenizer(product_prompt, return_tensors='pt')

output_ids = gpt2_model.generate(
    **inputs_gpt2,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    pad_token_id=gpt2_tokenizer.eos_token_id,
)
description = gpt2_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(description)


> **Output interpretation:** GPT-2 continues the prompt as auto-regressive text. `temperature=0.8` softens the probability distribution, making the output more creative than greedy (`temperature=1.0` is neutral; values below 0.7 become repetitive). `top_k=50` restricts sampling to the 50 most likely tokens at each step, preventing incoherent low-probability choices.


### 4.1 Effect of Temperature on Generation Diversity


In [ ]:
for temp in (0.3, 0.7, 1.2):
    ids = gpt2_model.generate(
        **inputs_gpt2,
        max_new_tokens=40,
        do_sample=True,
        temperature=temp,
        top_k=50,
        pad_token_id=gpt2_tokenizer.eos_token_id,
    )
    text = gpt2_tokenizer.decode(ids[0], skip_special_tokens=True)
    print(f'temperature={temp}:')
    print(text[len(product_prompt):])
    print()


> **Output interpretation:** Low temperature (0.3) produces near-deterministic, repetitive output — safe but dull. High temperature (1.2) injects noise that can produce novel phrasing but also grammatical errors. Temperature 0.7–0.9 is the practical sweet spot for creative business copy that remains coherent.


---

## 5. Business Case 3 — Automated Customer Service Responses with T5

**Scenario:** Generate templated but natural-sounding responses to common customer queries using T5 translation mode as a paraphrase engine.


In [ ]:
customer_queries = [
    'I cannot log in to my online banking account.',
    'My card was declined at the supermarket.',
    'I want to increase my credit card limit.',
]

for query in customer_queries:
    prompt = f'answer customer question: {query}'
    ids_in = t5_tokenizer(prompt, return_tensors='pt', max_length=128, truncation=True)
    ids_out = t5_model.generate(**ids_in, max_length=80, num_beams=4, early_stopping=True)
    response = t5_tokenizer.decode(ids_out[0], skip_special_tokens=True)
    print(f'Q: {query}')
    print(f'A: {response}')
    print()


> **Output interpretation:** T5-small is not specifically fine-tuned on customer service Q&A, so the responses may be generic or inaccurate. For production, fine-tune T5 on a dataset of `(query, ideal_response)` pairs from your own support ticket system. The task prefix `answer customer question:` guides T5 toward the generation mode but does not guarantee domain-specific accuracy.


---

## 6. Decoding Strategy Comparison

Different decoding strategies trade off between output quality, diversity, and computational cost.

| Strategy | Key parameter | Trade-off |
|---|---|---|
| **Greedy** | — | Fast; often repetitive |
| **Beam search** | `num_beams` | More fluent; less diverse |
| **Top-k sampling** | `top_k` | Creative; may be incoherent |
| **Top-p (nucleus)** | `top_p` | Dynamic vocab; better coherence than top-k |


In [ ]:
report_input = t5_tokenizer(
    'summarize: ' + financial_report,
    return_tensors='pt', max_length=512, truncation=True,
)

strategies = {
    'greedy': {'max_length': 60},
    'beam_search': {'max_length': 60, 'num_beams': 4, 'early_stopping': True},
    'top_k': {'max_length': 60, 'do_sample': True, 'top_k': 50, 'temperature': 0.8},
    'nucleus (top_p)': {'max_length': 60, 'do_sample': True, 'top_p': 0.92, 'temperature': 0.8},
}

for name, kwargs in strategies.items():
    ids = t5_model.generate(**report_input, **kwargs)
    text = t5_tokenizer.decode(ids[0], skip_special_tokens=True)
    print(f'[{name}] {text}')
    print()


> **Output interpretation:** Beam search typically produces the most grammatical output. Nucleus sampling (top-p) selects from the smallest set of tokens whose cumulative probability exceeds `top_p` — adapting the vocabulary size dynamically per step. This often outperforms fixed top-k in diversity while maintaining coherence. For deterministic, auditable business outputs (legal summaries, financial reports), prefer beam search. For creative copy, prefer nucleus sampling.


---

## 7. End-to-End Pipeline — Generation + Automatic Evaluation


In [ ]:
bleu_metric = evaluate.load('sacrebleu')
rouge_metric = evaluate.load('rouge')

reference_summary = (
    'Bancolombia reported a 15% increase in net income in Q3 2024, '
    'with digital transactions reaching 78% and a dividend of 250 pesos per share.'
)

generated_ids = t5_model.generate(**report_input, max_length=80, num_beams=4, early_stopping=True)
generated_summary = t5_tokenizer.decode(generated_ids[0], skip_special_tokens=True)

bleu = bleu_metric.compute(predictions=[generated_summary], references=[[reference_summary]])
rouge = rouge_metric.compute(predictions=[generated_summary], references=[reference_summary])

print(f'Generated: {generated_summary}')
print(f'Reference: {reference_summary}')
print(f'BLEU:    {bleu["score"]:.2f}')
print(f'ROUGE-1: {rouge["rouge1"]:.4f}')
print(f'ROUGE-L: {rouge["rougeL"]:.4f}')


> **Output interpretation:** BLEU and ROUGE measure n-gram overlap between the generated summary and the reference. Low scores do not necessarily mean poor quality — the model may use synonyms or paraphrase correctly. For a more reliable signal, evaluate against multiple reference summaries and use BERTScore (semantic similarity) alongside BLEU/ROUGE. In a production pipeline, log these metrics per document to track model drift over time.


---

## Summary

| Task | Model | Why |
|---|---|---|
| Report summarisation | T5 | Trained on seq2seq summarisation tasks |
| Creative product copy | GPT-2 | Auto-regressive; temperature controls creativity |
| Customer service Q&A | T5 (fine-tuned) | Structured input-output pairs |
| Quality evaluation | BLEU + ROUGE + BERTScore | Multi-metric coverage |
